In [ ]:
# -*- coding: utf-8 -*-
"""3D U-Net with Performance Tracking, SSA Analysis, and Apple-to-Apple Comparison"""

# Install required packages
!pip install torch torchvision torchaudio --quiet
!pip install tifffile scikit-image scipy matplotlib pandas --quiet
!pip install psutil gputil --quiet

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.filters import threshold_otsu, threshold_multiotsu
from scipy import stats
from scipy.ndimage import binary_opening, binary_closing
import pandas as pd
from datetime import datetime
import time
import psutil
import warnings
warnings.filterwarnings('ignore')

from tifffile import imread, imwrite


# Try to import GPUtil
try:
    import GPUtil
    GPU_AVAILABLE = True
except:
    GPU_AVAILABLE = False
    print("⚠️ GPUtil not available")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

NOISY_PATH = '/content/drive/MyDrive/Dataset TA/#1: DRP Noisy-not noisy-3x scale/noisy_1.tif'
CLEAN_PATH = '/content/drive/MyDrive/Dataset TA/#1: DRP Noisy-not noisy-3x scale/Not-noisy.tif'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configuration
class Config:
    NOISY_PATH = NOISY_PATH
    CLEAN_PATH = CLEAN_PATH
    PATCH_SIZE = 64
    NUM_PATCHES_PER_SLICE = 10
    INIT_FEATURES = 32
    BATCH_SIZE = 16
    EPOCHS = 100
    LEARNING_RATE = 5e-4
    PATIENCE = 15
    LR_PATIENCE = 10
    LR_FACTOR = 0.5
    INFERENCE_OVERLAP = 16
    VOXEL_SIZE_UM = 5.22
    TARGET_BIT_DEPTH = 8
    NORMALIZE_TO_UINT8 = True

config = Config()

# ============================================================================
# HARDWARE MONITOR & PERFORMANCE TRACKER
# ============================================================================

class HardwareMonitor:
    def __init__(self):
        self.process = psutil.Process()
        self.gpu_available = torch.cuda.is_available()
        self.gputil_available = GPU_AVAILABLE

    def get_all_stats(self):
        ram = psutil.virtual_memory()
        stats = {
            'cpu_percent': psutil.cpu_percent(interval=0.1),
            'ram': {
                'used_gb': ram.used / (1024**3),
                'total_gb': ram.total / (1024**3),
                'percent': ram.percent
            },
            'process_ram_gb': self.process.memory_info().rss / (1024**3)
        }

        if self.gpu_available:
            stats['gpu'] = {
                'allocated_gb': torch.cuda.memory_allocated() / (1024**3),
                'reserved_gb': torch.cuda.memory_reserved() / (1024**3),
                'max_allocated_gb': torch.cuda.max_memory_allocated() / (1024**3)
            }

            if self.gputil_available:
                try:
                    gpus = GPUtil.getGPUs()
                    if gpus:
                        gpu = gpus[0]
                        stats['gpu'].update({
                            'total_gb': gpu.memoryTotal / 1024,
                            'used_gb': gpu.memoryUsed / 1024,
                            'utilization_percent': gpu.memoryUtil * 100
                        })
                except:
                    pass

        return stats

    def print_stats(self, label="Current"):
        stats = self.get_all_stats()
        print(f"\n{'='*70}")
        print(f"{label} Hardware Usage")
        print(f"{'='*70}")
        print(f"  CPU: {stats['cpu_percent']:.1f}%")
        print(f"  RAM: {stats['ram']['used_gb']:.2f}/{stats['ram']['total_gb']:.2f} GB")
        print(f"  Process RAM: {stats['process_ram_gb']:.2f} GB")

        if 'gpu' in stats:
            print(f"  GPU Allocated: {stats['gpu']['allocated_gb']:.2f} GB")
            print(f"  GPU Peak: {stats['gpu']['max_allocated_gb']:.2f} GB")


class PerformanceTracker:
    def __init__(self):
        self.times = {}
        self.hardware_stats = {}
        self.current_phase = None
        self.start_time = None
        self.hw_monitor = HardwareMonitor()

    def start(self, phase_name):
        self.current_phase = phase_name
        self.start_time = time.time()
        self.hardware_stats[phase_name] = {'start': self.hw_monitor.get_all_stats()}
        print(f"\n⏱️  Starting: {phase_name}...")
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def end(self):
        if self.current_phase and self.start_time:
            elapsed = time.time() - self.start_time
            self.times[self.current_phase] = elapsed
            self.hardware_stats[self.current_phase]['end'] = self.hw_monitor.get_all_stats()
            self.hardware_stats[self.current_phase]['duration'] = elapsed

            if torch.cuda.is_available():
                self.hardware_stats[self.current_phase]['peak_gpu_gb'] = \
                    torch.cuda.max_memory_allocated() / (1024**3)

            print(f"✓ Completed: {self.current_phase} ({elapsed:.2f}s)")
            self.current_phase = None
            self.start_time = None
            return elapsed
        return 0

    def print_summary(self):
        print("\n" + "="*70)
        print("PERFORMANCE SUMMARY")
        print("="*70)
        total = sum(self.times.values())
        for phase, duration in self.times.items():
            pct = (duration / total) * 100
            print(f"  {phase:<40} {duration:>10.2f}s ({pct:>5.1f}%)")
        print(f"  {'TOTAL':<40} {total:>10.2f}s (100.0%)")

    def get_peak_memory_usage(self):
        peak = {'peak_gpu_gb': 0, 'peak_ram_gb': 0, 'peak_process_ram_gb': 0}
        for stats in self.hardware_stats.values():
            if 'peak_gpu_gb' in stats:
                peak['peak_gpu_gb'] = max(peak['peak_gpu_gb'], stats['peak_gpu_gb'])
            if 'end' in stats:
                peak['peak_ram_gb'] = max(peak['peak_ram_gb'],
                                         stats['end']['ram']['used_gb'])
                peak['peak_process_ram_gb'] = max(peak['peak_process_ram_gb'],
                                                 stats['end']['process_ram_gb'])
        return peak

perf_tracker = PerformanceTracker()
hw_monitor = HardwareMonitor()
hw_monitor.print_stats("Initial")

# ============================================================================
# NORMALIZATION FUNCTIONS
# ============================================================================

def convert_to_uint8(volume):
    v_min, v_max = volume.min(), volume.max()
    if v_max - v_min < 1e-8:
        return np.zeros_like(volume, dtype=np.uint8)
    normalized = (volume - v_min) / (v_max - v_min)
    return (normalized * 255).astype(np.uint8)

def standardize_volumes_for_comparison(noisy, denoised, ground_truth, bit_depth=8):
    if bit_depth == 8:
        noisy_std = convert_to_uint8(noisy)
        denoised_std = convert_to_uint8(denoised)
        truth_std = convert_to_uint8(ground_truth)
        data_range = 255
    else:
        v_min, v_max = 0, 1
        noisy_std = ((noisy - noisy.min()) / (noisy.max() - noisy.min())).astype(np.float32)
        denoised_std = ((denoised - denoised.min()) / (denoised.max() - denoised.min())).astype(np.float32)
        truth_std = ((ground_truth - ground_truth.min()) / (ground_truth.max() - ground_truth.min())).astype(np.float32)
        data_range = 1.0

    print(f"\n✓ Standardized to {bit_depth}-bit for fair comparison")
    return noisy_std, denoised_std, truth_std, data_range

# ============================================================================
# POROSITY & SSA FUNCTIONS
# ============================================================================

def calculate_porosity(volume, method='otsu', manual_threshold=None):
    if volume.dtype == np.uint8:
        volume_norm = volume.astype(np.float32) / 255.0
    else:
        volume_norm = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

    if method == 'otsu':
        threshold_value = threshold_otsu(volume_norm)
    elif method == 'manual' and manual_threshold is not None:
        threshold_value = manual_threshold
    else:
        threshold_value = threshold_otsu(volume_norm)

    binary_volume = volume_norm < threshold_value
    porosity = np.sum(binary_volume) / binary_volume.size * 100

    return porosity, binary_volume, threshold_value


def calculate_specific_surface_area(binary_volume, voxel_size_um=1.0):
    from skimage import measure

    if binary_volume.dtype != bool:
        binary_volume = binary_volume.astype(bool)

    try:
        verts, faces, normals, values = measure.marching_cubes(
            binary_volume.astype(np.float32),
            level=0.5,
            spacing=(voxel_size_um, voxel_size_um, voxel_size_um)
        )

        v0 = verts[faces[:, 0]]
        v1 = verts[faces[:, 1]]
        v2 = verts[faces[:, 2]]

        cross = np.cross(v1 - v0, v2 - v0)
        areas = 0.5 * np.sqrt(np.sum(cross**2, axis=1))
        surface_area_um2 = np.sum(areas)

    except Exception as e:
        print(f"  Warning: Marching cubes failed, using fallback")
        from scipy import ndimage
        struct = ndimage.generate_binary_structure(3, 1)
        dilated = ndimage.binary_dilation(binary_volume, structure=struct)
        boundary = dilated & ~binary_volume
        n_boundary_voxels = np.sum(boundary)
        surface_area_um2 = n_boundary_voxels * (voxel_size_um ** 2)

    depth, height, width = binary_volume.shape
    volume_um3 = depth * height * width * (voxel_size_um ** 3)
    ssa = surface_area_um2 / volume_um3 if volume_um3 > 0 else 0

    return ssa, surface_area_um2, volume_um3


def analyze_ssa_per_slice(binary_volume, voxel_size_um=1.0):
    from skimage import measure
    n_slices = binary_volume.shape[0]
    ssa_per_slice = []

    for i in range(n_slices):
        slice_2d = binary_volume[i].astype(bool)

        try:
            contours = measure.find_contours(slice_2d.astype(float), 0.5)
            perimeter_pixels = sum(len(c) for c in contours)
            perimeter_um = perimeter_pixels * voxel_size_um
        except:
            from scipy import ndimage
            struct = ndimage.generate_binary_structure(2, 1)
            dilated = ndimage.binary_dilation(slice_2d, structure=struct)
            boundary = dilated & ~slice_2d
            perimeter_pixels = np.sum(boundary)
            perimeter_um = perimeter_pixels * voxel_size_um

        area_um2 = np.sum(slice_2d) * (voxel_size_um ** 2)
        ssa_2d = perimeter_um / area_um2 if area_um2 > 0 else 0
        ssa_per_slice.append(ssa_2d)

    return ssa_per_slice

# ============================================================================
# LOAD DATA
# ============================================================================

perf_tracker.start("Data Loading")

print("\n" + "="*70)
print("LOADING DATA")
print("="*70)

noisy_volume = imread(config.NOISY_PATH).astype(np.float32)
clean_volume = imread(config.CLEAN_PATH).astype(np.float32)

print(f"Volume shape: {noisy_volume.shape}")
n_slices = noisy_volume.shape[0]

train_indices, val_indices = train_test_split(
    np.arange(n_slices), test_size=0.3, random_state=42
)

print(f"Train: {len(train_indices)}, Val: {len(val_indices)} slices")

perf_tracker.end()
hw_monitor.print_stats("After Data Loading")

# ============================================================================
# DATASET & MODEL
# ============================================================================

class VolumeDataset3D(Dataset):
    def __init__(self, noisy_volume, clean_volume, slice_indices, patch_size, num_patches):
        self.noisy_volume = (noisy_volume - noisy_volume.min()) / \
                           (noisy_volume.max() - noisy_volume.min() + 1e-8)
        self.clean_volume = (clean_volume - clean_volume.min()) / \
                           (clean_volume.max() - clean_volume.min() + 1e-8)
        self.vol_shape = noisy_volume.shape
        self.patch_size = patch_size
        self.num_patches = num_patches

    def __len__(self):
        return self.num_patches

    def __getitem__(self, idx):
        d, h, w = self.vol_shape
        ps = self.patch_size

        if d >= ps and h >= ps and w >= ps:
            d_start = np.random.randint(0, d - ps + 1)
            h_start = np.random.randint(0, h - ps + 1)
            w_start = np.random.randint(0, w - ps + 1)

            noisy_patch = self.noisy_volume[d_start:d_start+ps,
                                           h_start:h_start+ps,
                                           w_start:w_start+ps]
            clean_patch = self.clean_volume[d_start:d_start+ps,
                                           h_start:h_start+ps,
                                           w_start:w_start+ps]
        else:
            noisy_patch = self.noisy_volume
            clean_patch = self.clean_volume

        noisy_patch = torch.from_numpy(noisy_patch[np.newaxis, ...])
        clean_patch = torch.from_numpy(clean_patch[np.newaxis, ...])

        return noisy_patch, clean_patch


class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, init_features=32):
        super().__init__()
        f = init_features

        self.enc1 = DoubleConv3D(in_ch, f)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = DoubleConv3D(f, f*2)
        self.pool2 = nn.MaxPool3d(2)
        self.enc3 = DoubleConv3D(f*2, f*4)
        self.pool3 = nn.MaxPool3d(2)
        self.enc4 = DoubleConv3D(f*4, f*8)
        self.pool4 = nn.MaxPool3d(2)

        self.bottleneck = DoubleConv3D(f*8, f*16)

        self.up4 = nn.ConvTranspose3d(f*16, f*8, 2, stride=2)
        self.dec4 = DoubleConv3D(f*16, f*8)
        self.up3 = nn.ConvTranspose3d(f*8, f*4, 2, stride=2)
        self.dec3 = DoubleConv3D(f*8, f*4)
        self.up2 = nn.ConvTranspose3d(f*4, f*2, 2, stride=2)
        self.dec2 = DoubleConv3D(f*4, f*2)
        self.up1 = nn.ConvTranspose3d(f*2, f, 2, stride=2)
        self.dec1 = DoubleConv3D(f*2, f)

        self.out = nn.Conv3d(f, out_ch, 1)

    def _crop_and_concat(self, up, bypass):
        _, _, d_up, h_up, w_up = up.shape
        _, _, d_by, h_by, w_by = bypass.shape

        dd, dh, dw = (d_by-d_up)//2, (h_by-h_up)//2, (w_by-w_up)//2

        if dd > 0 or dh > 0 or dw > 0:
            bypass = bypass[:, :,
                          max(0,dd):dd+d_up if dd>0 else d_by,
                          max(0,dh):dh+h_up if dh>0 else h_by,
                          max(0,dw):dw+w_up if dw>0 else w_by]

        return torch.cat([up, bypass], dim=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.dec4(self._crop_and_concat(self.up4(b), e4))
        d3 = self.dec3(self._crop_and_concat(self.up3(d4), e3))
        d2 = self.dec2(self._crop_and_concat(self.up2(d3), e2))
        d1 = self.dec1(self._crop_and_concat(self.up1(d2), e1))

        return self.out(d1)


# Training functions
def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss = 0

    for noisy, clean in loader:
        noisy, clean = noisy.to(device), clean.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            output = model(noisy)
            loss = criterion(output, clean)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for noisy, clean in loader:
            noisy, clean = noisy.to(device), clean.to(device)
            with torch.cuda.amp.autocast():
                output = model(noisy)
                loss = criterion(output, clean)
            total_loss += loss.item()

    return total_loss / len(loader)


def denoise_volume_3d(model, noisy_volume, device, patch_size=64, overlap=16):
    model.eval()

    print(f"\nDenoising volume: {noisy_volume.shape}")

    noisy_min, noisy_max = noisy_volume.min(), noisy_volume.max()
    noisy_norm = (noisy_volume - noisy_min) / (noisy_max - noisy_min + 1e-8)

    d, h, w = noisy_volume.shape
    stride = patch_size - overlap

    d_patches = int(np.ceil(d / stride))
    h_patches = int(np.ceil(h / stride))
    w_patches = int(np.ceil(w / stride))

    d_padded = (d_patches - 1) * stride + patch_size
    h_padded = (h_patches - 1) * stride + patch_size
    w_padded = (w_patches - 1) * stride + patch_size

    pad_d, pad_h, pad_w = d_padded - d, h_padded - h, w_padded - w

    print(f"  Padded: {d_padded}×{h_padded}×{w_padded}")
    print(f"  Total patches: {d_patches*h_patches*w_patches}")

    noisy_padded = np.pad(noisy_norm, ((0,pad_d), (0,pad_h), (0,pad_w)), mode='reflect')
    denoised_padded = np.zeros_like(noisy_padded, dtype=np.float32)
    weight_padded = np.zeros_like(noisy_padded, dtype=np.float32)

    patch_count = 0
    total_patches = d_patches * h_patches * w_patches

    for d_idx in range(d_patches):
        d_start = d_idx * stride
        d_end = d_start + patch_size

        for h_idx in range(h_patches):
            h_start = h_idx * stride
            h_end = h_start + patch_size

            for w_idx in range(w_patches):
                w_start = w_idx * stride
                w_end = w_start + patch_size

                patch = noisy_padded[d_start:d_end, h_start:h_end, w_start:w_end]

                if patch.shape != (patch_size, patch_size, patch_size):
                    continue

                patch_tensor = torch.from_numpy(patch[np.newaxis, np.newaxis, ...]).to(device)

                with torch.no_grad():
                    with torch.cuda.amp.autocast():
                        denoised_patch = model(patch_tensor)

                denoised_patch = denoised_patch.squeeze().cpu().numpy()

                denoised_padded[d_start:d_end, h_start:h_end, w_start:w_end] += denoised_patch
                weight_padded[d_start:d_end, h_start:h_end, w_start:w_end] += 1.0

                patch_count += 1

                if patch_count % 10 == 0:
                    print(f"  Progress: {patch_count}/{total_patches} ({patch_count/total_patches*100:.1f}%)")

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    denoised_padded = denoised_padded / (weight_padded + 1e-8)
    denoised_volume = denoised_padded[:d, :h, :w]
    denoised_volume = denoised_volume * (noisy_max - noisy_min) + noisy_min

    print("✓ Denoising complete!")
    return denoised_volume


# ============================================================================
# BUILD MODEL & TRAIN
# ============================================================================

perf_tracker.start("Model Initialization")

print("\n" + "="*70)
print("BUILDING MODEL")
print("="*70)

train_dataset = VolumeDataset3D(noisy_volume, clean_volume, train_indices,
                                config.PATCH_SIZE,
                                len(train_indices) * config.NUM_PATCHES_PER_SLICE)
val_dataset = VolumeDataset3D(noisy_volume, clean_volume, val_indices,
                              config.PATCH_SIZE,
                              len(val_indices) * config.NUM_PATCHES_PER_SLICE)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                         shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE,
                       shuffle=False, num_workers=2, pin_memory=True)

model = UNet3D(in_ch=1, out_ch=1, init_features=config.INIT_FEATURES).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                 patience=config.LR_PATIENCE,
                                                 factor=config.LR_FACTOR)
scaler = torch.cuda.amp.GradScaler()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

perf_tracker.end()
hw_monitor.print_stats("After Model Init")

# ============================================================================
# TRAINING
# ============================================================================

perf_tracker.start("Training Phase")

print("\n" + "="*70)
print("TRAINING PHASE")
print("="*70)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'learning_rates': []}
patience_counter = 0
epoch_times = []

for epoch in range(config.EPOCHS):
    epoch_start = time.time()

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['learning_rates'].append(optimizer.param_groups[0]['lr'])

    scheduler.step(val_loss)

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)

    print(f"Epoch {epoch+1:3d}/{config.EPOCHS} - Train: {train_loss:.6f}, "
          f"Val: {val_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.2e}, "
          f"Time: {epoch_time:.2f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_3dunet_model.pth')
        print(f"  ✓ Best model saved")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= config.PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

training_time = perf_tracker.end()
hw_monitor.print_stats("After Training")

# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Training Loss', linewidth=2)
plt.plot(history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('3D U-Net Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('training_curves_3dunet.png', dpi=300, bbox_inches='tight')
plt.show()

model.load_state_dict(torch.load('best_3dunet_model.pth'))

# ============================================================================
# DENOISING
# ============================================================================

perf_tracker.start("Volume Denoising")

print("\n" + "="*70)
print("DENOISING VOLUME")
print("="*70)

denoised_volume = denoise_volume_3d(model, noisy_volume, device,
                                   config.PATCH_SIZE, config.INFERENCE_OVERLAP)

perf_tracker.end()
hw_monitor.print_stats("After Denoising")

# Save original denoised volume
print("\nSaving denoised volume...")
imwrite('denoised_volume_original_3dunet.tif', denoised_volume.astype(np.float32))
print("✓ Original denoised volume saved")

# ============================================================================
# STANDARDIZE & CALCULATE METRICS
# ============================================================================

perf_tracker.start("Image Quality Analysis")

print("\nStandardizing volumes for fair comparison...")
noisy_std, denoised_std, truth_std, data_range = standardize_volumes_for_comparison(
    noisy_volume, denoised_volume, clean_volume, bit_depth=config.TARGET_BIT_DEPTH
)

# Save standardized volumes
print("\nSaving standardized volumes...")
imwrite('standardized_noisy_3dunet.tif', noisy_std)
imwrite('standardized_denoised_3dunet.tif', denoised_std)
imwrite('standardized_truth_3dunet.tif', truth_std)
print("✓ Standardized volumes saved")

print("\n" + "="*70)
print("CALCULATING IMAGE QUALITY METRICS")
print("="*70)

def calculate_metrics(original, denoised, data_range=1.0):
    mse = np.mean((original.astype(np.float64) - denoised.astype(np.float64)) ** 2)

    if original.ndim == 2:
        psnr_val = psnr(original, denoised, data_range=data_range)
        ssim_val = ssim(original, denoised, data_range=data_range)
    else:
        psnr_vals = []
        ssim_vals = []
        for i in range(original.shape[0]):
            psnr_vals.append(psnr(original[i], denoised[i], data_range=data_range))
            ssim_vals.append(ssim(original[i], denoised[i], data_range=data_range))
        psnr_val = np.mean(psnr_vals)
        ssim_val = np.mean(ssim_vals)

    return {'MSE': mse, 'PSNR': psnr_val, 'SSIM': ssim_val}

metrics_before_all = []
metrics_after_all = []

for slice_idx in range(n_slices):
    noisy_slice = noisy_std[slice_idx]
    denoised_slice = denoised_std[slice_idx]
    truth_slice = truth_std[slice_idx]

    metrics_before = calculate_metrics(truth_slice, noisy_slice, data_range=data_range)
    metrics_after = calculate_metrics(truth_slice, denoised_slice, data_range=data_range)

    metrics_before_all.append(metrics_before)
    metrics_after_all.append(metrics_after)

avg_psnr_before = np.mean([m['PSNR'] for m in metrics_before_all])
avg_psnr_after = np.mean([m['PSNR'] for m in metrics_after_all])
avg_ssim_before = np.mean([m['SSIM'] for m in metrics_before_all])
avg_ssim_after = np.mean([m['SSIM'] for m in metrics_after_all])
avg_mse_before = np.mean([m['MSE'] for m in metrics_before_all])
avg_mse_after = np.mean([m['MSE'] for m in metrics_after_all])

print(f"\nBEFORE Denoising:")
print(f"  PSNR: {avg_psnr_before:.2f} dB")
print(f"  SSIM: {avg_ssim_before:.4f}")

print(f"\nAFTER 3D U-Net:")
print(f"  PSNR: {avg_psnr_after:.2f} dB (+{avg_psnr_after - avg_psnr_before:.2f})")
print(f"  SSIM: {avg_ssim_after:.4f} (+{avg_ssim_after - avg_ssim_before:.4f})")

psnr_improvements = [after['PSNR'] - before['PSNR']
                     for before, after in zip(metrics_before_all, metrics_after_all)]
ssim_improvements = [after['SSIM'] - before['SSIM']
                     for before, after in zip(metrics_before_all, metrics_after_all)]

t_stat_psnr, p_value_psnr = stats.ttest_1samp(psnr_improvements, 0)
t_stat_ssim, p_value_ssim = stats.ttest_1samp(ssim_improvements, 0)

print(f"\nStatistical Significance:")
print(f"  PSNR: t={t_stat_psnr:.4f}, p={p_value_psnr:.6f}")
print(f"  SSIM: t={t_stat_ssim:.4f}, p={p_value_ssim:.6f}")

perf_tracker.end()

# ============================================================================
# POROSITY ANALYSIS
# ============================================================================

perf_tracker.start("Porosity Analysis")

print("\n" + "="*70)
print("POROSITY ANALYSIS")
print("="*70)

sample_slice_idx = n_slices // 2
sample_slice = truth_std[sample_slice_idx]

if truth_std.dtype == np.uint8:
    sample_norm = sample_slice.astype(np.float32) / 255.0
else:
    sample_norm = (sample_slice - sample_slice.min()) / (sample_slice.max() - sample_slice.min() + 1e-8)

otsu_thresh = threshold_otsu(sample_norm)
print(f"Otsu threshold: {otsu_thresh:.4f}")

threshold_method = 'otsu'
manual_threshold_value = None

print("\nCalculating porosity from standardized volumes...")
porosity_noisy, binary_noisy, thresh_noisy = calculate_porosity(
    noisy_std, method=threshold_method, manual_threshold=manual_threshold_value
)

porosity_denoised, binary_denoised, thresh_denoised = calculate_porosity(
    denoised_std, method=threshold_method, manual_threshold=manual_threshold_value
)

porosity_truth, binary_truth, thresh_truth = calculate_porosity(
    truth_std, method=threshold_method, manual_threshold=manual_threshold_value
)

porosity_per_slice_noisy = []
porosity_per_slice_denoised = []
porosity_per_slice_truth = []

for slice_idx in range(n_slices):
    por_noisy = np.sum(binary_noisy[slice_idx]) / binary_noisy[slice_idx].size * 100
    por_denoised = np.sum(binary_denoised[slice_idx]) / binary_denoised[slice_idx].size * 100
    por_truth = np.sum(binary_truth[slice_idx]) / binary_truth[slice_idx].size * 100

    porosity_per_slice_noisy.append(por_noisy)
    porosity_per_slice_denoised.append(por_denoised)
    porosity_per_slice_truth.append(por_truth)

print("\n" + "="*70)
print("POROSITY RESULTS")
print("="*70)

print(f"\n{'Volume':<25} {'Threshold':<12} {'Porosity (%)':<15}")
print("-" * 70)
print(f"{'Ground Truth':<25} {thresh_truth:<12.4f} {porosity_truth:<15.2f}")
print(f"{'Noisy':<25} {thresh_noisy:<12.4f} {porosity_noisy:<15.2f}")
print(f"{'Denoised (3D U-Net)':<25} {thresh_denoised:<12.4f} {porosity_denoised:<15.2f}")

noisy_error = abs(porosity_noisy - porosity_truth)
denoised_error = abs(porosity_denoised - porosity_truth)
improvement = ((noisy_error - denoised_error) / noisy_error * 100) if noisy_error > 0 else 0

print(f"\n{'Comparison':<25} {'Error vs Truth':<25}")
print("-" * 70)
print(f"{'Noisy Error':<25} {noisy_error:>8.2f}%")
print(f"{'Denoised Error':<25} {denoised_error:>8.2f}%")
print(f"{'Error Reduction':<25} {improvement:>8.2f}%")

# Save binary volumes
imwrite('binary_truth_3dunet.tif', binary_truth.astype(np.uint8) * 255)
imwrite('binary_noisy_3dunet.tif', binary_noisy.astype(np.uint8) * 255)
imwrite('binary_denoised_3dunet.tif', binary_denoised.astype(np.uint8) * 255)
print("\n✓ Binary volumes saved")

perf_tracker.end()

# ============================================================================
# SPECIFIC SURFACE AREA ANALYSIS
# ============================================================================

perf_tracker.start("Specific Surface Area Analysis")

print("\n" + "="*70)
print("SPECIFIC SURFACE AREA (SSA) ANALYSIS")
print("="*70)

print(f"\nVoxel size: {config.VOXEL_SIZE_UM} μm")

print("\nCalculating overall SSA (3D analysis)...")
ssa_truth, sa_truth, vol_truth = calculate_specific_surface_area(
    binary_truth, config.VOXEL_SIZE_UM
)
print(f"  ✓ Ground Truth SSA calculated")

ssa_noisy, sa_noisy, vol_noisy = calculate_specific_surface_area(
    binary_noisy, config.VOXEL_SIZE_UM
)
print(f"  ✓ Noisy SSA calculated")

ssa_denoised, sa_denoised, vol_denoised = calculate_specific_surface_area(
    binary_denoised, config.VOXEL_SIZE_UM
)
print(f"  ✓ Denoised SSA calculated")

print(f"\n{'Volume':<25} {'Porosity (%)':<15} {'SSA (μm⁻¹)':<20} {'Surface Area (μm²)':<20}")
print("-" * 85)
print(f"{'Ground Truth':<25} {porosity_truth:<15.2f} {ssa_truth:<20.6f} {sa_truth:<20.2f}")
print(f"{'Noisy':<25} {porosity_noisy:<15.2f} {ssa_noisy:<20.6f} {sa_noisy:<20.2f}")
print(f"{'Denoised (3D U-Net)':<25} {porosity_denoised:<15.2f} {ssa_denoised:<20.6f} {sa_denoised:<20.2f}")

ssa_error_noisy = abs(ssa_noisy - ssa_truth)
ssa_error_denoised = abs(ssa_denoised - ssa_truth)
ssa_improvement = ((ssa_error_noisy - ssa_error_denoised) / ssa_error_noisy * 100) if ssa_error_noisy > 0 else 0

print(f"\n{'Comparison':<25} {'SSA Error (μm⁻¹)':<25} {'Error Reduction':<20}")
print("-" * 70)
print(f"{'Noisy Error':<25} {ssa_error_noisy:>8.6f} {'':<20}")
print(f"{'Denoised Error':<25} {ssa_error_denoised:>8.6f} {ssa_improvement:>19.2f}%")

print("\nCalculating per-slice SSA (2D analysis)...")
ssa_per_slice_truth = analyze_ssa_per_slice(binary_truth, config.VOXEL_SIZE_UM)
ssa_per_slice_noisy = analyze_ssa_per_slice(binary_noisy, config.VOXEL_SIZE_UM)
ssa_per_slice_denoised = analyze_ssa_per_slice(binary_denoised, config.VOXEL_SIZE_UM)

ssa_error_per_slice_noisy = [abs(n - t) for n, t in zip(ssa_per_slice_noisy, ssa_per_slice_truth)]
ssa_error_per_slice_denoised = [abs(d - t) for d, t in zip(ssa_per_slice_denoised, ssa_per_slice_truth)]

mean_ssa_error_noisy = np.mean(ssa_error_per_slice_noisy)
std_ssa_error_noisy = np.std(ssa_error_per_slice_noisy)
mean_ssa_error_denoised = np.mean(ssa_error_per_slice_denoised)
std_ssa_error_denoised = np.std(ssa_error_per_slice_denoised)

print(f"\nPer-Slice Statistics (2D SSA):")
print(f"  Mean SSA Error (Noisy):    {mean_ssa_error_noisy:.6f} ± {std_ssa_error_noisy:.6f} μm⁻¹")
print(f"  Mean SSA Error (Denoised): {mean_ssa_error_denoised:.6f} ± {std_ssa_error_denoised:.6f} μm⁻¹")

t_stat_ssa, p_value_ssa = stats.ttest_rel(ssa_error_per_slice_noisy, ssa_error_per_slice_denoised)

print(f"\nPaired t-test (Noisy vs Denoised SSA errors):")
print(f"  t-statistic: {t_stat_ssa:.4f}")
print(f"  p-value: {p_value_ssa:.6f}")

if p_value_ssa < 0.001:
    print(f"  ✓ SSA error reduction is HIGHLY SIGNIFICANT (p < 0.001)")
elif p_value_ssa < 0.05:
    print(f"  ✓ SSA error reduction is SIGNIFICANT (p < 0.05)")
else:
    print(f"  ✗ SSA error reduction is NOT significant (p >= 0.05)")

corr_ssa_noisy = np.corrcoef(ssa_per_slice_noisy, ssa_per_slice_truth)[0, 1]
corr_ssa_denoised = np.corrcoef(ssa_per_slice_denoised, ssa_per_slice_truth)[0, 1]

print(f"\nCorrelation with Ground Truth:")
print(f"  Noisy SSA:    r = {corr_ssa_noisy:.4f}")
print(f"  Denoised SSA: r = {corr_ssa_denoised:.4f}")
print(f"  Improvement: {(corr_ssa_denoised - corr_ssa_noisy):.4f}")

perf_tracker.end()

# ============================================================================
# VISUALIZATIONS
# ============================================================================

# Plot SSA per slice
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

slices = range(n_slices)

axes[0].plot(slices, ssa_per_slice_truth, label='Ground Truth',
            linewidth=2.5, alpha=0.9, color='green')
axes[0].plot(slices, ssa_per_slice_noisy, label='Noisy',
            linewidth=2, alpha=0.7, linestyle='--', color='red')
axes[0].plot(slices, ssa_per_slice_denoised, label='Denoised (3D U-Net)',
            linewidth=2, alpha=0.7, color='blue')
axes[0].axhline(y=np.mean(ssa_per_slice_truth), color='green', linestyle=':', alpha=0.5,
               label=f'Avg Truth: {np.mean(ssa_per_slice_truth):.6f} μm⁻¹')
axes[0].axhline(y=np.mean(ssa_per_slice_denoised), color='blue', linestyle=':', alpha=0.5,
               label=f'Avg Denoised: {np.mean(ssa_per_slice_denoised):.6f} μm⁻¹')
axes[0].set_xlabel('Slice Index')
axes[0].set_ylabel('Specific Surface Area (μm⁻¹)')
axes[0].set_title('Specific Surface Area per Slice (3D U-Net)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(slices, ssa_error_per_slice_noisy, label='Noisy Error',
            linewidth=2, alpha=0.7, color='red')
axes[1].plot(slices, ssa_error_per_slice_denoised, label='Denoised Error',
            linewidth=2, alpha=0.7, color='blue')
axes[1].fill_between(slices, ssa_error_per_slice_noisy, ssa_error_per_slice_denoised,
                     alpha=0.2, color='green', label='Improvement Region')
axes[1].set_xlabel('Slice Index')
axes[1].set_ylabel('Absolute SSA Error (μm⁻¹)')
axes[1].set_title('Specific Surface Area Error per Slice (3D U-Net)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('specific_surface_area_analysis_3dunet.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# EXPORT TO CSV
# ============================================================================

print("\n" + "="*70)
print("EXPORTING RESULTS TO CSV")
print("="*70)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# CSV 1: Combined per-slice analysis
df_combined = pd.DataFrame({
    'Slice_Index': range(n_slices),
    'PSNR_Before_dB': [m['PSNR'] for m in metrics_before_all],
    'PSNR_After_dB': [m['PSNR'] for m in metrics_after_all],
    'PSNR_Improvement_dB': [after['PSNR'] - before['PSNR']
                            for before, after in zip(metrics_before_all, metrics_after_all)],
    'SSIM_Before': [m['SSIM'] for m in metrics_before_all],
    'SSIM_After': [m['SSIM'] for m in metrics_after_all],
    'SSIM_Improvement': [after['SSIM'] - before['SSIM']
                         for before, after in zip(metrics_before_all, metrics_after_all)],
    'MSE_Before': [m['MSE'] for m in metrics_before_all],
    'MSE_After': [m['MSE'] for m in metrics_after_all],
    'Porosity_Ground_Truth_%': porosity_per_slice_truth,
    'Porosity_Noisy_%': porosity_per_slice_noisy,
    'Porosity_Denoised_%': porosity_per_slice_denoised,
    'Porosity_Error_Noisy_%': [abs(n - t) for n, t in zip(porosity_per_slice_noisy, porosity_per_slice_truth)],
    'Porosity_Error_Denoised_%': [abs(d - t) for d, t in zip(porosity_per_slice_denoised, porosity_per_slice_truth)],
    'SSA_Truth_um_inv': ssa_per_slice_truth,
    'SSA_Noisy_um_inv': ssa_per_slice_noisy,
    'SSA_Denoised_um_inv': ssa_per_slice_denoised,
    'SSA_Error_Noisy_um_inv': ssa_error_per_slice_noisy,
    'SSA_Error_Denoised_um_inv': ssa_error_per_slice_denoised,
})

csv1 = f'combined_analysis_with_SSA_3dunet_{timestamp}.csv'
df_combined.to_csv(csv1, index=False, float_format='%.8f')
print(f"✓ Saved: {csv1}")

# CSV 2: Overall summary with performance
perf_summary = perf_tracker.get_summary()
peak_memory = perf_tracker.get_peak_memory_usage()

summary_data = {
    'Metric': [
        'Volume_Depth', 'Volume_Height', 'Volume_Width', 'Total_Voxels',
        'Original_Noisy_dtype', 'Original_Denoised_dtype', 'Original_Truth_dtype',
        'Standardized_Bit_Depth', 'Standardized_dtype', 'Data_Range_For_Metrics',
        'Threshold_Method', 'Threshold_Ground_Truth', 'Threshold_Noisy', 'Threshold_Denoised',
        'Porosity_Ground_Truth_%', 'Porosity_Noisy_%', 'Porosity_Denoised_%',
        'Porosity_Error_Noisy_%', 'Porosity_Error_Denoised_%', 'Porosity_Error_Reduction_%',
        'Voxel_Size_um',
        'SSA_3D_Truth_um_inv', 'SSA_3D_Noisy_um_inv', 'SSA_3D_Denoised_um_inv',
        'SSA_3D_Error_Noisy_um_inv', 'SSA_3D_Error_Denoised_um_inv', 'SSA_3D_Error_Reduction_%',
        'Surface_Area_Truth_um2', 'Surface_Area_Noisy_um2', 'Surface_Area_Denoised_um2',
        'SSA_2D_Mean_Truth_um_inv', 'SSA_2D_Mean_Noisy_um_inv', 'SSA_2D_Mean_Denoised_um_inv',
        'PSNR_Before_Mean_dB', 'PSNR_After_Mean_dB', 'PSNR_Improvement_dB',
        'SSIM_Before_Mean', 'SSIM_After_Mean', 'SSIM_Improvement',
        'MSE_Before_Mean', 'MSE_After_Mean',
        'PSNR_t_statistic', 'PSNR_p_value',
        'SSIM_t_statistic', 'SSIM_p_value',
        'Porosity_t_statistic', 'Porosity_p_value',
        'SSA_t_statistic', 'SSA_p_value',
        'Porosity_Correlation_Noisy', 'Porosity_Correlation_Denoised',
        'SSA_Correlation_Noisy', 'SSA_Correlation_Denoised',
        'Model_Total_Parameters', 'Model_Architecture',
        'Training_Epochs', 'Best_Validation_Loss', 'Avg_Epoch_Time_s',
        'Time_Data_Loading_s', 'Time_Model_Init_s', 'Time_Training_s',
        'Time_Denoising_s', 'Time_Image_Quality_Analysis_s',
        'Time_Porosity_Analysis_s', 'Time_SSA_Analysis_s', 'Time_Total_s',
        'Peak_GPU_Memory_GB', 'Peak_RAM_Usage_GB', 'Peak_Process_RAM_GB',
        'System_Total_RAM_GB', 'GPU_Name', 'CPU_Count'
    ],
    'Value': [
        noisy_volume.shape[0], noisy_volume.shape[1], noisy_volume.shape[2], noisy_volume.size,
        str(noisy_volume.dtype), str(denoised_volume.dtype), str(clean_volume.dtype),
        config.TARGET_BIT_DEPTH, str(noisy_std.dtype), data_range,
        threshold_method, thresh_truth, thresh_noisy, thresh_denoised,
        porosity_truth, porosity_noisy, porosity_denoised,
        noisy_error, denoised_error, improvement,
        config.VOXEL_SIZE_UM,
        ssa_truth, ssa_noisy, ssa_denoised,
        ssa_error_noisy, ssa_error_denoised, ssa_improvement,
        sa_truth, sa_noisy, sa_denoised,
        np.mean(ssa_per_slice_truth), np.mean(ssa_per_slice_noisy), np.mean(ssa_per_slice_denoised),
        avg_psnr_before, avg_psnr_after, avg_psnr_after - avg_psnr_before,
        avg_ssim_before, avg_ssim_after, avg_ssim_after - avg_ssim_before,
        avg_mse_before, avg_mse_after,
        t_stat_psnr, p_value_psnr,
        t_stat_ssim, p_value_ssim,
        stats.ttest_rel([abs(n - t) for n, t in zip(porosity_per_slice_noisy, porosity_per_slice_truth)],
                       [abs(d - t) for d, t in zip(porosity_per_slice_denoised, porosity_per_slice_truth)])[0],
        stats.ttest_rel([abs(n - t) for n, t in zip(porosity_per_slice_noisy, porosity_per_slice_truth)],
                       [abs(d - t) for d, t in zip(porosity_per_slice_denoised, porosity_per_slice_truth)])[1],
        t_stat_ssa, p_value_ssa,
        np.corrcoef(porosity_per_slice_noisy, porosity_per_slice_truth)[0, 1],
        np.corrcoef(porosity_per_slice_denoised, porosity_per_slice_truth)[0, 1],
        corr_ssa_noisy, corr_ssa_denoised,
        total_params, '3D U-Net',
        len(history['train_loss']), best_val_loss, np.mean(epoch_times),
        perf_summary['individual'].get('Data Loading', 0),
        perf_summary['individual'].get('Model Initialization', 0),
        perf_summary['individual'].get('Training Phase', 0),
        perf_summary['individual'].get('Volume Denoising', 0),
        perf_summary['individual'].get('Image Quality Analysis', 0),
        perf_summary['individual'].get('Porosity Analysis', 0),
        perf_summary['individual'].get('Specific Surface Area Analysis', 0),
        perf_summary['total'],
        peak_memory['peak_gpu_gb'],
        peak_memory['peak_ram_gb'],
        peak_memory['peak_process_ram_gb'],
        psutil.virtual_memory().total / (1024**3),
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Only',
        psutil.cpu_count()
    ]
}

df_summary = pd.DataFrame(summary_data)
csv2 = f'overall_summary_with_performance_3dunet_{timestamp}.csv'
df_summary.to_csv(csv2, index=False, float_format='%.8f')
print(f"✓ Saved: {csv2}")

# CSV 3: Performance breakdown
hardware_per_phase = []
for phase in perf_summary['individual'].keys():
    phase_data = {
        'Phase': phase,
        'Time_seconds': perf_summary['individual'][phase],
        'Percentage': perf_summary['individual'][phase]/perf_summary['total']*100
    }

    if phase in perf_summary['hardware']:
        hw_stats = perf_summary['hardware'][phase]
        if 'peak_gpu_gb' in hw_stats:
            phase_data['Peak_GPU_Memory_GB'] = hw_stats['peak_gpu_gb']
        if 'start' in hw_stats and 'end' in hw_stats:
            phase_data['RAM_Start_GB'] = hw_stats['start']['process_ram_gb']
            phase_data['RAM_End_GB'] = hw_stats['end']['process_ram_gb']
            phase_data['RAM_Delta_GB'] = hw_stats['end']['process_ram_gb'] - hw_stats['start']['process_ram_gb']

    hardware_per_phase.append(phase_data)

df_performance = pd.DataFrame(hardware_per_phase)
csv3 = f'performance_breakdown_with_hardware_3dunet_{timestamp}.csv'
df_performance.to_csv(csv3, index=False, float_format='%.4f')
print(f"✓ Saved: {csv3}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

perf_tracker.print_summary()
hw_monitor.print_stats("Final")

print("\n" + "="*70)
print("3D U-NET COMPLETE ANALYSIS FINISHED")
print("="*70)

print(f"\n📊 FINAL SUMMARY:")
print(f"  Architecture: 3D U-Net")
print(f"  Parameters: {total_params:,}")
print(f"  Total Time: {perf_summary['total']:.2f}s")
print(f"  Peak GPU: {peak_memory['peak_gpu_gb']:.2f} GB")
print(f"  Peak RAM: {peak_memory['peak_process_ram_gb']:.2f} GB")

print(f"\n📈 IMAGE QUALITY:")
print(f"  PSNR: {avg_psnr_before:.2f} → {avg_psnr_after:.2f} dB (+{avg_psnr_after - avg_psnr_before:.2f})")
print(f"  SSIM: {avg_ssim_before:.4f} → {avg_ssim_after:.4f} (+{avg_ssim_after - avg_ssim_before:.4f})")

print(f"\n🔬 POROSITY:")
print(f"  Ground Truth: {porosity_truth:.2f}%")
print(f"  Error Reduction: {improvement:.1f}%")

print(f"\n📐 SPECIFIC SURFACE AREA:")
print(f"  Ground Truth (3D): {ssa_truth:.6f} μm⁻¹")
print(f"  Noisy Error: {ssa_error_noisy:.6f} μm⁻¹")
print(f"  Denoised Error: {ssa_error_denoised:.6f} μm⁻¹")
print(f"  Improvement: {ssa_improvement:.1f}%")

print(f"\n💾 OUTPUT FILES:")
print(f"  • best_3dunet_model.pth")
print(f"  • denoised_volume_original_3dunet.tif")
print(f"  • standardized_*_3dunet.tif (3 files)")
print(f"  • binary_*_3dunet.tif (3 files)")  # Menutup baris yang terpotong sebelumnya

if torch.cuda.is_available():
    print(f"\n🖥️  GPU USAGE:")
    print(f"  Max Memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    print(f"  Current Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\n" + "="*70)
print("✅ 3D U-NET ANALYSIS COMPLETED SUCCESSFULLY!")
print("="*70)

# ============================================================================
# ZIPPING & DOWNLOADING RESULTS
# ============================================================================

print("\n" + "="*70)
print("PREPARING DOWNLOADS")
print("="*70)

import zipfile
from google.colab import files

# Nama file ZIP dengan timestamp
zip_filename = f'Full_Analysis_Results_3DUNET_{timestamp}.zip'

print(f"Creating archive: {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # 1. Masukkan File CSV (Analisis & Performance)
    csv_files = [csv1, csv2, csv3]
    for csv_file in csv_files:
        if os.path.exists(csv_file):
            zipf.write(csv_file)
            print(f"  ➕ Added: {csv_file}")

    # 2. Masukkan Grafik/Plot
    plot_files = [
        'training_curves_3dunet.png',
        'specific_surface_area_analysis_3dunet.png'
    ]
    for plot in plot_files:
        if os.path.exists(plot):
            zipf.write(plot)
            print(f"  ➕ Added: {plot}")

    # 3. Masukkan Volume Denoised (PENTING)
    # a. Versi Asli (Float32)
    if os.path.exists('denoised_volume_original_3dunet.tif'):
        zipf.write('denoised_volume_original_3dunet.tif')
        print(f"  ➕ Added: Original Denoised Volume (Float32)")

    # b. Versi Apple-to-Apple (Standardized Uint8)
    if os.path.exists('standardized_denoised_3dunet.tif'):
        zipf.write('standardized_denoised_3dunet.tif')
        print(f"  ➕ Added: Standardized Denoised Volume (Apple-to-Apple/Uint8)")

    # 4. Masukkan Model Terbaik
    if os.path.exists('best_3dunet_model.pth'):
        zipf.write('best_3dunet_model.pth')
        print(f"  ➕ Added: Best Model Checkpoint")

    # 5. Masukkan Binary Masks (Opsional, untuk validasi porositas)
    binary_files = [
        'binary_truth_3dunet.tif',
        'binary_noisy_3dunet.tif',
        'binary_denoised_3dunet.tif'
    ]
    for bf in binary_files:
        if os.path.exists(bf):
            zipf.write(bf)
            print(f"  ➕ Added: {bf}")

print(f"\n✓ Archive created successfully ({os.path.getsize(zip_filename)/1e6:.2f} MB)")

# Download otomatis
print("\nInitiating download...")
try:
    files.download(zip_filename)
    print("✅ Download started automatically!")
except Exception as e:
    print(f"⚠️ Automatic download failed. Please download '{zip_filename}' manually from the Files panel.")
    print(f"Error: {str(e)}")

# Opsional: Download file penting secara terpisah jika ZIP gagal atau terlalu besar
print("\n(Backup Link) Downloading key CSV summary...")
try:
    files.download(csv2) # Download summary CSV separately just in case
except:
    pass

In [ ]:
# ============================================================================
# 1. BAGIAN PERBAIKAN (PATCH)
# ============================================================================
import types
import pandas as pd
import os
import psutil
import torch
import zipfile
from google.colab import files
from datetime import datetime

# Fungsi yang hilang kita definisikan manual di sini
def get_summary_patch(self):
    total = sum(self.times.values())
    return {
        'total': total,
        'individual': self.times,
        'hardware': self.hardware_stats
    }

# Kita tempelkan fungsi ini ke objek 'perf_tracker' yang sudah ada di memori
# Jadi tidak perlu training ulang
perf_tracker.get_summary = types.MethodType(get_summary_patch, perf_tracker)

print("✅ Perbaikan berhasil diterapkan ke 'perf_tracker'. Melanjutkan proses export...")

# ============================================================================
# 2. LANJUTAN PROSES YANG TADINYA ERROR (EXPORT CSV & ZIP)
# ============================================================================

# Ambil timestamp baru atau gunakan yang lama jika variabel masih ada
try:
    timestamp
except NameError:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# --- LANJUTAN KODE DARI BARIS 1000 ---

# CSV 2: Overall summary with performance
perf_summary = perf_tracker.get_summary() # Sekarang ini tidak akan error lagi
peak_memory = perf_tracker.get_peak_memory_usage()

summary_data = {
    'Metric': [
        'Volume_Depth', 'Volume_Height', 'Volume_Width', 'Total_Voxels',
        'Original_Noisy_dtype', 'Original_Denoised_dtype', 'Original_Truth_dtype',
        'Standardized_Bit_Depth', 'Standardized_dtype', 'Data_Range_For_Metrics',
        'Threshold_Method', 'Threshold_Ground_Truth', 'Threshold_Noisy', 'Threshold_Denoised',
        'Porosity_Ground_Truth_%', 'Porosity_Noisy_%', 'Porosity_Denoised_%',
        'Porosity_Error_Noisy_%', 'Porosity_Error_Denoised_%', 'Porosity_Error_Reduction_%',
        'Voxel_Size_um',
        'SSA_3D_Truth_um_inv', 'SSA_3D_Noisy_um_inv', 'SSA_3D_Denoised_um_inv',
        'SSA_3D_Error_Noisy_um_inv', 'SSA_3D_Error_Denoised_um_inv', 'SSA_3D_Error_Reduction_%',
        'Surface_Area_Truth_um2', 'Surface_Area_Noisy_um2', 'Surface_Area_Denoised_um2',
        'SSA_2D_Mean_Truth_um_inv', 'SSA_2D_Mean_Noisy_um_inv', 'SSA_2D_Mean_Denoised_um_inv',
        'PSNR_Before_Mean_dB', 'PSNR_After_Mean_dB', 'PSNR_Improvement_dB',
        'SSIM_Before_Mean', 'SSIM_After_Mean', 'SSIM_Improvement',
        'MSE_Before_Mean', 'MSE_After_Mean',
        'PSNR_t_statistic', 'PSNR_p_value',
        'SSIM_t_statistic', 'SSIM_p_value',
        'Porosity_t_statistic', 'Porosity_p_value',
        'SSA_t_statistic', 'SSA_p_value',
        'Porosity_Correlation_Noisy', 'Porosity_Correlation_Denoised',
        'SSA_Correlation_Noisy', 'SSA_Correlation_Denoised',
        'Model_Total_Parameters', 'Model_Architecture',
        'Training_Epochs', 'Best_Validation_Loss', 'Avg_Epoch_Time_s',
        'Time_Data_Loading_s', 'Time_Model_Init_s', 'Time_Training_s',
        'Time_Denoising_s', 'Time_Image_Quality_Analysis_s',
        'Time_Porosity_Analysis_s', 'Time_SSA_Analysis_s', 'Time_Total_s',
        'Peak_GPU_Memory_GB', 'Peak_RAM_Usage_GB', 'Peak_Process_RAM_GB',
        'System_Total_RAM_GB', 'GPU_Name', 'CPU_Count'
    ],
    'Value': [
        noisy_volume.shape[0], noisy_volume.shape[1], noisy_volume.shape[2], noisy_volume.size,
        str(noisy_volume.dtype), str(denoised_volume.dtype), str(clean_volume.dtype),
        config.TARGET_BIT_DEPTH, str(noisy_std.dtype), data_range,
        threshold_method, thresh_truth, thresh_noisy, thresh_denoised,
        porosity_truth, porosity_noisy, porosity_denoised,
        noisy_error, denoised_error, improvement,
        config.VOXEL_SIZE_UM,
        ssa_truth, ssa_noisy, ssa_denoised,
        ssa_error_noisy, ssa_error_denoised, ssa_improvement,
        sa_truth, sa_noisy, sa_denoised,
        np.mean(ssa_per_slice_truth), np.mean(ssa_per_slice_noisy), np.mean(ssa_per_slice_denoised),
        avg_psnr_before, avg_psnr_after, avg_psnr_after - avg_psnr_before,
        avg_ssim_before, avg_ssim_after, avg_ssim_after - avg_ssim_before,
        avg_mse_before, avg_mse_after,
        t_stat_psnr, p_value_psnr,
        t_stat_ssim, p_value_ssim,
        stats.ttest_rel([abs(n - t) for n, t in zip(porosity_per_slice_noisy, porosity_per_slice_truth)],
                        [abs(d - t) for d, t in zip(porosity_per_slice_denoised, porosity_per_slice_truth)])[0],
        stats.ttest_rel([abs(n - t) for n, t in zip(porosity_per_slice_noisy, porosity_per_slice_truth)],
                        [abs(d - t) for d, t in zip(porosity_per_slice_denoised, porosity_per_slice_truth)])[1],
        t_stat_ssa, p_value_ssa,
        np.corrcoef(porosity_per_slice_noisy, porosity_per_slice_truth)[0, 1],
        np.corrcoef(porosity_per_slice_denoised, porosity_per_slice_truth)[0, 1],
        corr_ssa_noisy, corr_ssa_denoised,
        total_params, '3D U-Net',
        len(history['train_loss']), best_val_loss, np.mean(epoch_times),
        perf_summary['individual'].get('Data Loading', 0),
        perf_summary['individual'].get('Model Initialization', 0),
        perf_summary['individual'].get('Training Phase', 0),
        perf_summary['individual'].get('Volume Denoising', 0),
        perf_summary['individual'].get('Image Quality Analysis', 0),
        perf_summary['individual'].get('Porosity Analysis', 0),
        perf_summary['individual'].get('Specific Surface Area Analysis', 0),
        perf_summary['total'],
        peak_memory['peak_gpu_gb'],
        peak_memory['peak_ram_gb'],
        peak_memory['peak_process_ram_gb'],
        psutil.virtual_memory().total / (1024**3),
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Only',
        psutil.cpu_count()
    ]
}

df_summary = pd.DataFrame(summary_data)
csv2 = f'overall_summary_with_performance_3dunet_{timestamp}.csv'
df_summary.to_csv(csv2, index=False, float_format='%.8f')
print(f"✓ Saved: {csv2}")

# CSV 3: Performance breakdown
hardware_per_phase = []
for phase in perf_summary['individual'].keys():
    phase_data = {
        'Phase': phase,
        'Time_seconds': perf_summary['individual'][phase],
        'Percentage': perf_summary['individual'][phase]/perf_summary['total']*100
    }

    if phase in perf_summary['hardware']:
        hw_stats = perf_summary['hardware'][phase]
        if 'peak_gpu_gb' in hw_stats:
            phase_data['Peak_GPU_Memory_GB'] = hw_stats['peak_gpu_gb']
        if 'start' in hw_stats and 'end' in hw_stats:
            phase_data['RAM_Start_GB'] = hw_stats['start']['process_ram_gb']
            phase_data['RAM_End_GB'] = hw_stats['end']['process_ram_gb']
            phase_data['RAM_Delta_GB'] = hw_stats['end']['process_ram_gb'] - hw_stats['start']['process_ram_gb']

    hardware_per_phase.append(phase_data)

df_performance = pd.DataFrame(hardware_per_phase)
csv3 = f'performance_breakdown_with_hardware_3dunet_{timestamp}.csv'
df_performance.to_csv(csv3, index=False, float_format='%.4f')
print(f"✓ Saved: {csv3}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

perf_tracker.print_summary()
hw_monitor.print_stats("Final")

print("\n" + "="*70)
print("3D U-NET COMPLETE ANALYSIS FINISHED")
print("="*70)

print(f"\n📊 FINAL SUMMARY:")
print(f"  Architecture: 3D U-Net")
print(f"  Parameters: {total_params:,}")
print(f"  Total Time: {perf_summary['total']:.2f}s")
print(f"  Peak GPU: {peak_memory['peak_gpu_gb']:.2f} GB")
print(f"  Peak RAM: {peak_memory['peak_process_ram_gb']:.2f} GB")

print(f"\n📈 IMAGE QUALITY:")
print(f"  PSNR: {avg_psnr_before:.2f} -> {avg_psnr_after:.2f} dB (+{avg_psnr_after - avg_psnr_before:.2f})")
print(f"  SSIM: {avg_ssim_before:.4f} -> {avg_ssim_after:.4f} (+{avg_ssim_after - avg_ssim_before:.4f})")

print(f"\n🔬 POROSITY:")
print(f"  Ground Truth: {porosity_truth:.2f}%")
print(f"  Error Reduction: {improvement:.1f}%")

print(f"\n📐 SPECIFIC SURFACE AREA:")
print(f"  Ground Truth (3D): {ssa_truth:.6f} μm⁻¹")
print(f"  Noisy Error: {ssa_error_noisy:.6f} μm⁻¹")
print(f"  Denoised Error: {ssa_error_denoised:.6f} μm⁻¹")
print(f"  Improvement: {ssa_improvement:.1f}%")

print(f"\n💾 OUTPUT FILES:")
print(f"  • best_3dunet_model.pth")
print(f"  • denoised_volume_original_3dunet.tif")
print(f"  • standardized_*_3dunet.tif (3 files)")
print(f"  • binary_*_3dunet.tif (3 files)")

if torch.cuda.is_available():
    print(f"\n🖥️  GPU USAGE:")
    print(f"  Max Memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    print(f"  Current Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\n" + "="*70)
print("✅ 3D U-NET ANALYSIS COMPLETED SUCCESSFULLY!")
print("="*70)

# ============================================================================
# ZIPPING & DOWNLOADING RESULTS
# ============================================================================

print("\n" + "="*70)
print("PREPARING DOWNLOADS")
print("="*70)

# Nama file ZIP dengan timestamp
zip_filename = f'Full_Analysis_Results_3DUNET_{timestamp}.zip'

print(f"Creating archive: {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # 1. Masukkan File CSV (Analisis & Performance)
    # Gunakan nama file yang benar dari CSV1 yang sudah dibuat sebelumnya (jika ada di memori)
    # Jika csv1 belum ada di memori karena error, kita recreate namanya
    csv1_name = f'combined_analysis_with_SSA_3dunet_{timestamp}.csv'

    csv_files = [csv1_name, csv2, csv3]
    for csv_file in csv_files:
        if os.path.exists(csv_file):
            zipf.write(csv_file)
            print(f"  ➕ Added: {csv_file}")

    # 2. Masukkan Grafik/Plot
    plot_files = [
        'training_curves_3dunet.png',
        'specific_surface_area_analysis_3dunet.png'
    ]
    for plot in plot_files:
        if os.path.exists(plot):
            zipf.write(plot)
            print(f"  ➕ Added: {plot}")

    # 3. Masukkan Volume Denoised
    if os.path.exists('denoised_volume_original_3dunet.tif'):
        zipf.write('denoised_volume_original_3dunet.tif')
        print(f"  ➕ Added: Original Denoised Volume (Float32)")

    if os.path.exists('standardized_denoised_3dunet.tif'):
        zipf.write('standardized_denoised_3dunet.tif')
        print(f"  ➕ Added: Standardized Denoised Volume (Apple-to-Apple/Uint8)")

    # 4. Masukkan Model Terbaik
    if os.path.exists('best_3dunet_model.pth'):
        zipf.write('best_3dunet_model.pth')
        print(f"  ➕ Added: Best Model Checkpoint")

    # 5. Masukkan Binary Masks
    binary_files = [
        'binary_truth_3dunet.tif',
        'binary_noisy_3dunet.tif',
        'binary_denoised_3dunet.tif'
    ]
    for bf in binary_files:
        if os.path.exists(bf):
            zipf.write(bf)
            print(f"  ➕ Added: {bf}")

print(f"\n✓ Archive created successfully ({os.path.getsize(zip_filename)/1e6:.2f} MB)")

# Download otomatis
print("\nInitiating download...")
try:
    files.download(zip_filename)
    print("✅ Download started automatically!")
except Exception as e:
    print(f"⚠️ Automatic download failed. Please download '{zip_filename}' manually from the Files panel.")
    print(f"Error: {str(e)}")